In [1]:
%matplotlib inline

import sys
import yaml
import copy

import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../../')
sys.path.append('../../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.yolov11_pose.data.dataset import YOLODataset
from computer_vision.yolov11_pose.utils import DEFAULT_CFG_DICT
from computer_vision.yolov11_pose.parameter_parser import parser
from computer_vision.misc import alpha_bending, instance2mask
from computer_vision.yolov11_pose.cfg import get_cfg

In [2]:
from matplotlib import patches
import matplotlib.pyplot as plt

cmap = plt.get_cmap('tab10', 10)
plt.rcParams.update({'font.size'   : 12})

In [3]:
args=parser.parse_args('--deterministic'.split())

task='segment'
if task=='pose':
    data_dirpath='D:/data/ultralytics/coco8-pose'
    data_config='../../coco8-pose.yaml'
elif task=='detect':
    data_dirpath='D:/data/ultralytics/coco/images/train2017'
    data_config='../../coco.yaml'
elif task=='segment':
    data_dirpath='D:/data/ultralytics/coco8-seg'
    data_config='../../coco8-seg.yaml'
    
dataset=YOLODataset(args=args, data=data_config, task=task, img_path=data_dirpath, imgsz=640, cache=False,augment=True, 
                    hyp=DEFAULT_CFG_DICT, prefix='',rect=False,batch_size=16,stride=32,pad=0.5, single_cls=False,classes=None,
                fraction=1.,channels=3)
for i in range(4): print(dataset.labels[i]['cls'].shape)

hyp=get_cfg()

In data.dataset.YOLODataset.get_labels cache_path D:\data\ultralytics\coco8-seg\labels\train.cache
Scanning D:\data\ultralytics\coco8-seg\labels\train.cache... 8 images, 0 backgrounds, 0 corrupts
(8, 1)
(2, 1)
(2, 1)
(1, 1)


In [4]:
batch=[dataset[i] for i in range(5)]
print([b.keys() for b in batch])

[dict_keys(['im_file', 'ori_shape', 'resized_shape', 'masks', 'img', 'cls', 'bboxes', 'batch_idx']), dict_keys(['im_file', 'ori_shape', 'resized_shape', 'masks', 'img', 'cls', 'bboxes', 'batch_idx']), dict_keys(['im_file', 'ori_shape', 'resized_shape', 'masks', 'img', 'cls', 'bboxes', 'batch_idx']), dict_keys(['im_file', 'ori_shape', 'resized_shape', 'masks', 'img', 'cls', 'bboxes', 'batch_idx']), dict_keys(['im_file', 'ori_shape', 'resized_shape', 'masks', 'img', 'cls', 'bboxes', 'batch_idx'])]


In [5]:
new_batch={}
batch=[dict(sorted(b.items())) for b in batch] # make sure the keys are in the same order by sorting each dict by keys
keys=batch[0].keys()
values=list(zip(*[list(b.values()) for b in batch]))

print([b.keys() for b in batch])
print('\n', keys)

[dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape']), dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape']), dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape']), dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape']), dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape'])]

 dict_keys(['batch_idx', 'bboxes', 'cls', 'im_file', 'img', 'masks', 'ori_shape', 'resized_shape'])


In [10]:
for i, k in enumerate(keys):
    value=values[i]
    if k in {'img', 'text_feats'}:
        print(k , [v.shape for v in value], end=',')
        value=torch.stack(value, 0) # BxCxHxW
        print(value.shape)
    elif k=='visuals':
        value=torch.nn.utils.rnn.pad_sequence(value, batch_first=True)
    if k in {'masks', 'keypoints', 'bboxes', 'cls', 'segments', 'obb'}:
        print(k, [v.shape for v in value], end=',')
        value=torch.cat(value, 0)
        print(value.shape)
    new_batch[k]=value

bboxes [torch.Size([2, 4]), torch.Size([7, 4]), torch.Size([0, 4]), torch.Size([3, 4]), torch.Size([3, 4])],torch.Size([15, 4])
cls [torch.Size([2, 1]), torch.Size([7, 1]), torch.Size([0, 1]), torch.Size([3, 1]), torch.Size([3, 1])],torch.Size([15, 1])
img [torch.Size([3, 640, 640]), torch.Size([3, 640, 640]), torch.Size([3, 640, 640]), torch.Size([3, 640, 640]), torch.Size([3, 640, 640])],torch.Size([5, 3, 640, 640])
masks [torch.Size([1, 160, 160]), torch.Size([1, 160, 160]), torch.Size([1, 160, 160]), torch.Size([1, 160, 160]), torch.Size([1, 160, 160])],torch.Size([5, 160, 160])


In [11]:
new_batch

{'batch_idx': (tensor([0., 0.]),
  tensor([0., 0., 0., 0., 0., 0., 0.]),
  tensor([]),
  tensor([0., 0., 0.]),
  tensor([0., 0., 0.])),
 'bboxes': tensor([[0.2519, 0.5351, 0.4939, 0.4152],
         [0.9171, 0.4094, 0.1619, 0.4680],
         [0.8673, 0.6537, 0.2651, 0.2308],
         [0.8859, 0.6708, 0.2265, 0.1979],
         [0.8489, 0.4826, 0.2578, 0.1846],
         [0.8314, 0.2113, 0.1735, 0.2399],
         [0.3147, 0.2113, 0.1735, 0.2399],
         [0.3147, 0.5552, 0.1735, 0.2399],
         [0.9393, 0.4892, 0.1213, 0.1819],
         [0.1187, 0.7239, 0.2370, 0.4334],
         [0.8751, 0.1842, 0.2459, 0.3683],
         [0.1073, 0.2118, 0.2129, 0.4236],
         [0.2518, 0.7605, 0.5008, 0.4743],
         [0.3280, 0.2217, 0.4014, 0.4430],
         [0.2469, 0.1265, 0.4904, 0.2519]]),
 'cls': tensor([[45.],
         [58.],
         [45.],
         [50.],
         [45.],
         [23.],
         [23.],
         [23.],
         [45.],
         [58.],
         [58.],
         [22.],
        

In [12]:
new_batch['batch_idx']=list(new_batch['batch_idx'])
for i in range(len(new_batch['batch_idx'])):
    new_batch['batch_idx'][i]+=i # add target image index for build_targets()
new_batch['batch_idx']

[tensor([0., 0.]),
 tensor([1., 1., 1., 1., 1., 1., 1.]),
 tensor([]),
 tensor([3., 3., 3.]),
 tensor([4., 4., 4.])]

In [13]:
new_batch['batch_idx']=torch.cat(new_batch['batch_idx'], 0)
new_batch['batch_idx']

tensor([0., 0., 1., 1., 1., 1., 1., 1., 1., 3., 3., 3., 4., 4., 4.])